# 04 气象因素与负荷相关性分析

**题目四 城市电力负荷分析与预测** ｜ 负责：江玉婷（GitHub: Lizhuiii；规律分析 + 报告主笔）

| 项目 | 内容 |
|---|---|
| 上游数据 | `data/processed/power_load_cleaned.csv` |
| 本 notebook 产出 | 气象-负荷 Pearson/Spearman 相关表、全样本热力图、按月与月×小时拆解、分季节相关表 |
| 对应题目要求 | 研究气象因素与负荷的相关性，并用热力图展示 |
| 下游消费者 | 谭煜霖（成品图重绘）、江玉婷（报告第 6 节）、沈成源（03 特征工程参考） |

## 分析口径

Pearson 相关系数只描述线性关联，不等于因果。全样本相关会把季节、昼夜、节假日和生产安排混在一起，因此本 notebook 同时给出：

1. 全样本 Pearson 与 Spearman；
2. 按月温度-负荷相关；
3. 月×小时温度-负荷相关；
4. 分季节的气象-负荷相关表。

这样才能区分“全年平均关系”和“局部时段关系”，避免把某个季节的结论外推到全年。


## 步骤 0　环境与路径


In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)


def find_root(marker="data"):
    "从当前工作目录向上查找仓库根目录（与 01 保持一致）。"
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / marker / "raw").is_dir():
            return d
    raise FileNotFoundError(f"未在 {here} 及其父目录中找到 {marker}/raw")


ROOT = find_root()
PROCESSED = ROOT / "data" / "processed" / "power_load_cleaned.csv"
FIG_DIR = ROOT / "figures" / "patterns"
TABLE_DIR = ROOT / "tables"
for p in (FIG_DIR, TABLE_DIR):
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

ZONE_COLS = ["zone1_power_consumption", "zone2_power_consumption", "zone3_power_consumption"]
TOTAL = "zone_total_power_consumption"
WEATHER_COLS = ["temperature", "humidity", "wind_speed", "general_diffuse_flows", "diffuse_flows"]
VAR_LABEL = {
    "temperature": "温度",
    "humidity": "湿度",
    "wind_speed": "风速",
    "general_diffuse_flows": "general diffuse flows",
    "diffuse_flows": "diffuse flows",
    "zone1_power_consumption": "Zone1 负荷",
    "zone2_power_consumption": "Zone2 负荷",
    "zone3_power_consumption": "Zone3 负荷",
    "zone_total_power_consumption": "总负荷",
}
SEASON_ORDER = ["春季", "夏季", "秋季", "冬季"]
SEASON_MAP = {
    12: "冬季", 1: "冬季", 2: "冬季",
    3: "春季", 4: "春季", 5: "春季",
    6: "夏季", 7: "夏季", 8: "夏季",
    9: "秋季", 10: "秋季", 11: "秋季",
}

print(f"仓库根目录 : {ROOT}")
print(f"清洗数据   : {PROCESSED.name} (存在: {PROCESSED.exists()})")


仓库根目录 : C:\Users\Jyvting\Desktop\power-load-forecast-topic4
清洗数据   : power_load_cleaned.csv (存在: True)


## 步骤 1　加载清洗数据


In [2]:
df = pd.read_csv(PROCESSED, index_col="datetime", parse_dates=True)
df.index.name = "datetime"

print(f"形状   : {df.shape}")
print(f"频率   : {df.index.inferred_freq}")
print(f"范围   : {df.index[0]} -> {df.index[-1]}")
print(f"缺失   : {int(df.isna().sum().sum())}")
display(df.head(3))


形状   : (52416, 9)
频率   : 10min
范围   : 2017-01-01 00:00:00 -> 2017-12-30 23:50:00
缺失   : 0


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1_power_consumption,zone2_power_consumption,zone3_power_consumption,zone_total_power_consumption
datetime,,,,,,,,,
2017-01-01 00:00:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.875380,20240.96386,70425.535440
2017-01-01 00:10:00,6.414,74.5,0.083,0.070,0.085,29814.68354,17567.781155,20131.08434,67513.549035
2017-01-01 00:20:00,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.686930,19668.43373,67803.221930


## 步骤 2　全样本 Pearson 与 Spearman 相关

先给全样本相关矩阵，再抽取“气象变量 × 负荷”的聚焦子表。Pearson 看线性关联，Spearman 看单调关联的稳健性；两者方向一致时，结论更稳。


In [3]:
num_cols = WEATHER_COLS + ZONE_COLS + [TOTAL]
pearson = df[num_cols].corr(method="pearson")
spearman = df[num_cols].corr(method="spearman")

sub_cols = [TOTAL] + ZONE_COLS
pearson_sub = pearson.loc[WEATHER_COLS, sub_cols]
spearman_sub = spearman.loc[WEATHER_COLS, sub_cols]
pearson_sub.round(6).to_csv(TABLE_DIR / "weather_load_corr.csv", encoding="utf-8-sig", index_label="气象变量")
spearman_sub.round(6).to_csv(TABLE_DIR / "weather_load_corr_spearman.csv", encoding="utf-8-sig", index_label="气象变量")

print("=== Pearson：气象变量 × 负荷 ===")
display(pearson_sub.round(3).rename(index=VAR_LABEL, columns=VAR_LABEL))
print("=== Spearman：气象变量 × 负荷 ===")
display(spearman_sub.round(3).rename(index=VAR_LABEL, columns=VAR_LABEL))

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(pearson.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels([VAR_LABEL[c] for c in num_cols], rotation=45, ha="right", fontsize=9)
ax.set_yticklabels([VAR_LABEL[c] for c in num_cols], fontsize=9)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        v = pearson.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                color="white" if abs(v) > 0.5 else "black", fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pearson r")
ax.set_title("全样本气象-负荷相关系数热力图（Pearson）", loc="left")
fig.savefig(FIG_DIR / "figG_corr_heatmap.png", dpi=180)
plt.close(fig)

print(f"温度-总负荷 Pearson r = {pearson.loc['temperature', TOTAL]:.3f}")
print(f"湿度-总负荷 Pearson r = {pearson.loc['humidity', TOTAL]:.3f}")
print(f"风速-总负荷 Pearson r = {pearson.loc['wind_speed', TOTAL]:.3f}")


=== Pearson：气象变量 × 负荷 ===

,总负荷,Zone1 负荷,Zone2 负荷,Zone3 负荷
温度,0.488,0.440,0.382,0.490
湿度,-0.299,-0.287,-0.295,-0.233
风速,0.222,0.168,0.147,0.279
general diffuse flows,0.150,0.188,0.157,0.063
diffuse flows,0.032,0.080,0.045,-0.038


=== Spearman：气象变量 × 负荷 ===


,总负荷,Zone1 负荷,Zone2 负荷,Zone3 负荷
温度,0.469,0.433,0.379,0.436
湿度,-0.306,-0.300,-0.309,-0.212
风速,0.107,0.107,0.087,0.079
general diffuse flows,0.227,0.261,0.242,0.085
diffuse flows,0.082,0.125,0.097,-0.021


温度-总负荷 Pearson r = 0.488
湿度-总负荷 Pearson r = -0.299
风速-总负荷 Pearson r = 0.222


### 全样本相关结论

- 温度与总负荷的 Pearson 相关系数为 **0.488**，在五个气象变量中最强；湿度为 **-0.299**，风速为 **0.222**，两个辐照类变量分别为 0.150 和 0.032。Spearman 的方向与 Pearson 基本一致。
- 这只能说明“温度较高时负荷通常较高”，不能直接推出“温度每升高 1 度，负荷增加多少”，也不能把相关全部解释为空调制冷。
- 全样本相关不是“接近 0”。夏季和冬季都有正相关，只是强度不同；真正的方向变化出现在更细的月份×小时切片里，下一节会拆开看。


## 步骤 3　温度-负荷相关随月份变化

按月计算温度与总负荷、三个分区负荷的 Pearson 相关，检查“相关方向是否随季节反转”。


In [4]:
rows = []
for m in range(1, 13):
    sub = df[df.index.month == m]
    rows.append({
        "month": m,
        "total": sub["temperature"].corr(sub[TOTAL]),
        **{f"z{i + 1}": sub["temperature"].corr(sub[ZONE_COLS[i]]) for i in range(3)},
    })
monthly_temp_corr = pd.DataFrame(rows).set_index("month")
monthly_temp_corr.round(6).to_csv(TABLE_DIR / "temp_load_corr_month.csv", encoding="utf-8-sig")
print("=== 按月温度-负荷 Pearson r ===")
display(monthly_temp_corr.round(3))

fig, ax = plt.subplots(figsize=(10, 4.8))
bars = ax.bar(monthly_temp_corr.index, monthly_temp_corr["total"], color="#1f4e9c", alpha=0.85, label="总负荷")
for b, v in zip(bars, monthly_temp_corr["total"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.018, f"{v:.2f}", ha="center", fontsize=8)
for col, color, label in [("z1", "#e07b39", "Zone1"), ("z2", "#2ca25f", "Zone2"), ("z3", "#c0392b", "Zone3")]:
    ax.plot(monthly_temp_corr.index, monthly_temp_corr[col], marker="o", lw=1.5, color=color, label=label)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(range(1, 13))
ax.set_xlabel("月份")
ax.set_ylabel("温度-负荷 Pearson r")
ax.set_title("温度与负荷相关系数的月份变化", loc="left")
ax.legend(fontsize=8, ncol=4)
fig.savefig(FIG_DIR / "figH_temp_corr_by_month.png", dpi=180)
plt.close(fig)

print(f"最小月相关 : {monthly_temp_corr['total'].min():.3f}（{monthly_temp_corr['total'].idxmin()} 月）")
print(f"最大月相关 : {monthly_temp_corr['total'].max():.3f}（{monthly_temp_corr['total'].idxmax()} 月）")
print(f"夏季(6-8)平均 r : {monthly_temp_corr.loc[6:8, 'total'].mean():.3f}")
print(f"冬季(12,1,2)平均 r : {monthly_temp_corr.loc[[12, 1, 2], 'total'].mean():.3f}")
print(f"12 个月是否全部为正 : {bool((monthly_temp_corr['total'] > 0).all())}")


=== 按月温度-负荷 Pearson r ===


,total,z1,z2,z3
month,,,,
1,0.336,0.377,0.322,0.243
2,0.268,0.301,0.317,0.137
3,0.369,0.411,0.394,0.228
4,0.367,0.380,0.443,0.219
5,0.346,0.365,0.365,0.266
6,0.194,0.191,0.206,0.169
7,0.509,0.523,0.560,0.415
8,0.450,0.435,0.510,0.378
9,0.383,0.393,0.399,0.254


最小月相关 : 0.194（6 月）
最大月相关 : 0.509（7 月）
夏季(6-8)平均 r : 0.384
冬季(12,1,2)平均 r : 0.307
12 个月是否全部为正 : True


### 月份级结论

- 12 个月的温度-总负荷相关系数 **全部为正**，范围从 6 月的 0.194 到 7 月的 0.509；夏季平均约 0.384，冬季平均约 0.307。
- 因此不能写“夏季正相关、冬季负相关”。正确表述是：**相关方向在月度尺度上保持为正，但强度随季节变化，7 月最强、6 月最弱**。
- 月度汇总仍可能掩盖同一个月内白天与夜间、工作日与周末的差异，因此还需要看月×小时切片。


## 步骤 4　温度-负荷相关随小时变化

对每个月、每个小时分别计算温度与总负荷的 Pearson 相关。每个格子大约由 30 天 × 6 个 10 分钟观测构成，适合看日内结构，但仍会受天气过程和节假日影响。


In [5]:
pivot = pd.DataFrame(index=range(1, 13), columns=range(24), dtype=float)
for m in range(1, 13):
    for h in range(24):
        sub = df[(df.index.month == m) & (df.index.hour == h)]
        if len(sub) > 5:
            pivot.loc[m, h] = sub["temperature"].corr(sub[TOTAL])
pivot.index.name = "month"
pivot.columns.name = "hour"
pivot.round(6).to_csv(TABLE_DIR / "temp_load_corr_month_hour.csv", encoding="utf-8-sig")

negative_count = int((pivot < 0).sum().sum())
print(f"负相关格子 : {negative_count} / {pivot.size}")
print(f"最小 r     : {pivot.min().min():.3f} at {pivot.stack().idxmin()}")
print(f"最大 r     : {pivot.max().max():.3f} at {pivot.stack().idxmax()}")

stacked = pivot.stack().sort_values()
print("\n最负的 5 个 月×小时 切片：")
display(stacked.head(5).to_frame("Pearson r"))
print("最正的 5 个 月×小时 切片：")
display(stacked.tail(5).sort_values(ascending=False).to_frame("Pearson r"))

fig, ax = plt.subplots(figsize=(12, 5.2))
im = ax.imshow(pivot.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xlabel("小时")
ax.set_ylabel("月份")
ax.set_xticks(range(0, 24, 2))
ax.set_yticks(range(1, 13))
ax.set_title("温度-负荷相关系数的 月×小时 结构", loc="left")
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="Pearson r")
fig.savefig(FIG_DIR / "figI_temp_corr_month_hour.png", dpi=180)
plt.close(fig)


负相关格子 : 112 / 288
最小 r     : -0.463 at (5, 15)
最大 r     : 0.550 at (7, 9)

最负的 5 个 月×小时 切片：


Pearson r
month hour           
5     15    -0.463263
6     2     -0.454868
1     20    -0.423414
      18    -0.417244
5     16    -0.409383

最正的 5 个 月×小时 切片：


Pearson r
month hour           
7     9      0.549982
2     19     0.547269
10    9      0.524741
      8      0.508777
      7      0.503735

### 月×小时结论

- 288 个“月×小时”切片中有 112 个为负相关，说明温度与负荷的关系在更细的时段上并不稳定；最强正相关出现在 7 月上午（约 0.55），最强负相关出现在 5 月午后（约 -0.46）。
- 局部负相关不能直接解释为“冬季采暖导致低温高负荷”。它也可能来自天气过程、节假日、生产安排、空调使用时段变化等混杂因素。现有数据没有节假日、分区用户类型等字段，无法进一步分离。
- 对建模的含义是：**温度效应应允许随月份和小时变化**。如果 沈成源的 03 只放一个全局温度系数，容易把不同时段的正负关系平均掉；可考虑加入 `month × temperature`、`hour × temperature` 或分季节模型。


## 步骤 5　分季节的气象-负荷相关

把样本按季节分组，分别计算五个气象变量与总负荷的 Pearson 相关，检查全样本结论是否由某一季节主导。


In [6]:
season = df.index.month.map(SEASON_MAP)
season_rows = {}
for s in SEASON_ORDER:
    sub = df[season == s]
    season_rows[s] = sub[WEATHER_COLS + [TOTAL]].corr().loc[WEATHER_COLS, TOTAL]
season_corr = pd.DataFrame(season_rows).reindex(columns=SEASON_ORDER)
season_corr.index = [VAR_LABEL[c] for c in season_corr.index]
season_corr.round(6).to_csv(TABLE_DIR / "weather_load_corr_by_season.csv", encoding="utf-8-sig")
print("=== 分季节气象-总负荷 Pearson r ===")
display(season_corr.round(3))

print("\n各季节温度-总负荷 r：")
for s in SEASON_ORDER:
    print(f"  {s}: {season_corr.loc['温度', s]:.3f}")


=== 分季节气象-总负荷 Pearson r ===


,春季,夏季,秋季,冬季
温度,0.348,0.482,0.370,0.295
湿度,-0.280,-0.294,-0.228,-0.234
风速,0.069,0.185,-0.034,0.009
general diffuse flows,0.124,0.039,0.092,0.089
diffuse flows,0.039,-0.121,0.075,0.108



各季节温度-总负荷 r：
  春季: 0.348
  夏季: 0.482
  秋季: 0.370
  冬季: 0.295


### 分季节结论

- 温度在四个季节的相关方向都为正，但强度不同；这再次说明“冬季负相关”的旧表述不成立。
- 湿度在多数季节为负相关，风速多为正相关。风速的正相关很可能混入了季节和天气过程的影响，不能据此认为“风越大负荷越高”。
- 报告应把温度作为主要气象解释变量，把湿度和风速作为辅助变量，并明确相关不等于因果。


## 步骤 6　小结与交付清单

| 题目要求 | 本 notebook 对应 |
|---|---|
| 气象-负荷相关性 | 报告图 7、`tables/weather_load_corr.csv`、`tables/weather_load_corr_spearman.csv` |
| 相关性深入（报告加分项） | 报告图 8、报告图 9、`tables/temp_load_corr_month.csv`、`tables/temp_load_corr_month_hour.csv` |
| 稳健性检查 | `tables/weather_load_corr_by_season.csv` |

### 产出文件

| 路径 | 内容 |
|---|---|
| `figures/patterns/figG_corr_heatmap.png` | 全样本 Pearson 热力图 |
| `figures/patterns/figH_temp_corr_by_month.png` | 温度-负荷相关随月份变化 |
| `figures/patterns/figI_temp_corr_month_hour.png` | 温度-负荷相关 月×小时 热力图 |
| `tables/weather_load_corr.csv` | Pearson 聚焦子表 |
| `tables/weather_load_corr_spearman.csv` | Spearman 稳健性对照 |
| `tables/temp_load_corr_month.csv` | 按月温度相关 |
| `tables/temp_load_corr_month_hour.csv` | 月×小时温度相关 |
| `tables/weather_load_corr_by_season.csv` | 分季节气象相关 |

### 给报告和视频的固定口径

- 温度-总负荷全样本 r 为 **0.488**，是五个气象变量中最强的。
- 月度相关全部为正，强度在 0.194-0.509 之间；不要写“冬夏方向相反”。
- 月×小时矩阵存在局部负相关，但只能说明关系随时段变化，不能据此推断采暖因果。
- 相关分析不能替代回归系数或因果识别；沈成源的 03 若要解释温度影响，应报告模型设定和评估结果。
